# API Key

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
print(f'{os.environ['OPENAI_API_KEY'][:20]}...') 

sk-proj-iKU13YeoxNgF...


# import

In [2]:
import json

from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

# from langchain_openai.chat_models import ChatOpenAI
# from langchain_openai.llms.base import OpenAI
# from langchain_core.prompts.prompt import PromptTemplate
# from langchain_core.prompts.chat import ChatPromptTemplate

# ↓ 걍 아래와 같이 해도 된다.
from langchain_openai import OpenAI, ChatOpenAI
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate

from langchain.chat_models import init_chat_model

In [9]:
chat = ChatOpenAI(
    model="gpt-4o",
    temperature=0.1,
    streaming=True,
    callbacks=[
        StreamingStdOutCallbackHandler()
    ],
)

# PromptTemplate 포맷팅 방법2가지

In [4]:
# 방법1
t = PromptTemplate.from_template("What is the capital of {country}")
t.format(country="France")

'What is the capital of France'

In [5]:
# 방법2
t2 = PromptTemplate(
    template = "What is the capital of {country}",
    input_variables = ['country'],   # 입력변수들을 PromptTemplate 에 명시적으로 알려준다.
)

t2.format(country="Germany")


'What is the capital of Germany'

# FewShotPromptTemplate
모델에 예제(example) 추가

In [ ]:
# 모델에게 '어떻게 대답해야 하는 지에 대한 예제(example)'를 AI 모델에게 주는 것이
# prompt 를 사용해서 '어떻게 대답해야 하는지 알려주는 것'보다 훨씬 좋다

# FewShotPromptTemplate 이 하는 일이 바로 그거다!
# - 이를 통해 예제(샘플)를 형식화(포맷) 할수 있다.
# - 이런 예제들을 데이터베이스등에 저장시켜놓고 활용할수도 있다

In [6]:
from langchain_core.prompts.few_shot import FewShotPromptTemplate

In [7]:
# 예제들.
# 모델이 나에게 '이런식으로 답변해 줬으면 좋겠다' 라고 제시하는 example 들.
examples = [
  {
    "question": "What do you know about France?",

    # ↓ 원하는 형식의 답변 이다..
    "answer": """
      Here is what I know:
      Capital: Paris
      Language: French
      Food: Wine and Cheese
      Currency: Euro
      """,
  },
  {
    "question": "What do you know about Italy?",
    "answer": """
      I know this:
      Capital: Rome
      Language: Italian
      Food: Pizza and Pasta
      Currency: Euro
      """,
  },
  {
    "question": "What do you know about Greece?",
    "answer": """
      I know this:
      Capital: Athens
      Language: Greek
      Food: Souvlaki and Feta Cheese
      Currency: Euro
      """,
  },    
]

In [10]:
# 일단 예제 없이 모델 호출
chat.invoke("What do you know about France?")

France, officially known as the French Republic, is a country located in Western Europe. It is known for its rich history, cultural heritage, and significant influence on global affairs. Here are some key points about France:

1. **Geography**: France is the largest country in the European Union by area and is bordered by Belgium, Luxembourg, Germany, Switzerland, Italy, Monaco, Spain, and Andorra. It also has coastlines along the Atlantic Ocean and the Mediterranean Sea.

2. **Capital and Major Cities**: The capital of France is Paris, which is also its largest city and a major global center for art, fashion, and culture. Other major cities include Marseille, Lyon, Toulouse, Nice, and Bordeaux.

3. **Language**: The official language is French, which is also one of the most widely spoken languages in the world.

4. **Government**: France is a semi-presidential representative democratic republic. The President is the head of state, while the Prime Minister is the head of government.

5

AIMessage(content="France, officially known as the French Republic, is a country located in Western Europe. It is known for its rich history, cultural heritage, and significant influence on global affairs. Here are some key points about France:\n\n1. **Geography**: France is the largest country in the European Union by area and is bordered by Belgium, Luxembourg, Germany, Switzerland, Italy, Monaco, Spain, and Andorra. It also has coastlines along the Atlantic Ocean and the Mediterranean Sea.\n\n2. **Capital and Major Cities**: The capital of France is Paris, which is also its largest city and a major global center for art, fashion, and culture. Other major cities include Marseille, Lyon, Toulouse, Nice, and Bordeaux.\n\n3. **Language**: The official language is French, which is also one of the most widely spoken languages in the world.\n\n4. **Government**: France is a semi-presidential representative democratic republic. The President is the head of state, while the Prime Minister is

In [11]:
# 예제 사용

# 포맷 준비 <- examples 로 포맷팅 할거다
example_template = """
    Human: {question}
    AI: {answer}
"""
# ↑ {question} 과 {answer} 는 위 examples 의 샘플과 동일한 key 를 사용하여 작성.


In [13]:
example_prompt = PromptTemplate.from_template(example_template)

example_prompt

PromptTemplate(input_variables=['answer', 'question'], input_types={}, partial_variables={}, template='\n    Human: {question}\n    AI: {answer}\n')

In [16]:
# 데이터 한개로 포맷팅
print(example_prompt.format(**examples[2]))


    Human: What do you know about Greece?
    AI: 
      I know this:
      Capital: Athens
      Language: Greek
      Food: Souvlaki and Feta Cheese
      Currency: Euro
      



In [18]:
# 실제 prompt 작성
prompt = FewShotPromptTemplate(
    example_prompt=example_prompt,   # 사용할 prompt
    examples = examples,   # 준비된 예제들

    # ↑ 이렇게만 해도 랭체인이 알아서 각각의 예제 리스트들을
    #  위 prompt 를 사용하여 포맷팅(형식화) 합니다  (밑)

    # 여기에 유저의 질문을 넣어줄겁니다

    # suffix=
    # 형식화된 모든 예제 마지막에 나오는 내용.
    # 그래서 한번 다 형식화가 끝난 다음, 사용자의 질문은 무엇인지 나옵니다

    # 여기서는 위에 있는 동일한 형식을 사용할 겁니다    

    suffix="Human: What do you know about {country}?",
    input_variables=['country']
)

prompt

FewShotPromptTemplate(input_variables=['country'], input_types={}, partial_variables={}, examples=[{'question': 'What do you know about France?', 'answer': '\n      Here is what I know:\n      Capital: Paris\n      Language: French\n      Food: Wine and Cheese\n      Currency: Euro\n      '}, {'question': 'What do you know about Italy?', 'answer': '\n      I know this:\n      Capital: Rome\n      Language: Italian\n      Food: Pizza and Pasta\n      Currency: Euro\n      '}, {'question': 'What do you know about Greece?', 'answer': '\n      I know this:\n      Capital: Athens\n      Language: Greek\n      Food: Souvlaki and Feta Cheese\n      Currency: Euro\n      '}], example_prompt=PromptTemplate(input_variables=['answer', 'question'], input_types={}, partial_variables={}, template='\n    Human: {question}\n    AI: {answer}\n'), suffix='Human: What do you know about {country}?')

In [19]:
print(prompt.format(country="Germany"))


    Human: What do you know about France?
    AI: 
      Here is what I know:
      Capital: Paris
      Language: French
      Food: Wine and Cheese
      Currency: Euro
      



    Human: What do you know about Italy?
    AI: 
      I know this:
      Capital: Rome
      Language: Italian
      Food: Pizza and Pasta
      Currency: Euro
      



    Human: What do you know about Greece?
    AI: 
      I know this:
      Capital: Athens
      Language: Greek
      Food: Souvlaki and Feta Cheese
      Currency: Euro
      


Human: What do you know about Germany?


In [ ]:
# step1  example 리스트를 만들고   examples
# step2  FewShotPromptTemplate 에 전달했고 examples=
# step3  어떻게 전달한 예제들을 형식화 할지 알려주었고
# step4  마지막에 질문을 포함시켰다.  suffix, input_variables

# AI 는 우리의 예제들과 똑같은 구조, 형태로 답변하게 될겁니다


In [20]:
chain = prompt | chat

chain

FewShotPromptTemplate(input_variables=['country'], input_types={}, partial_variables={}, examples=[{'question': 'What do you know about France?', 'answer': '\n      Here is what I know:\n      Capital: Paris\n      Language: French\n      Food: Wine and Cheese\n      Currency: Euro\n      '}, {'question': 'What do you know about Italy?', 'answer': '\n      I know this:\n      Capital: Rome\n      Language: Italian\n      Food: Pizza and Pasta\n      Currency: Euro\n      '}, {'question': 'What do you know about Greece?', 'answer': '\n      I know this:\n      Capital: Athens\n      Language: Greek\n      Food: Souvlaki and Feta Cheese\n      Currency: Euro\n      '}], example_prompt=PromptTemplate(input_variables=['answer', 'question'], input_types={}, partial_variables={}, template='\n    Human: {question}\n    AI: {answer}\n'), suffix='Human: What do you know about {country}?')
| ChatOpenAI(callbacks=[<langchain_core.callbacks.streaming_stdout.StreamingStdOutCallbackHandler object at

In [21]:
chain.invoke({"country": "Germany"})

AI: 
  Here's what I know:
  Capital: Berlin
  Language: German
  Food: Bratwurst and Sauerkraut
  Currency: Euro

AIMessage(content="AI: \n  Here's what I know:\n  Capital: Berlin\n  Language: German\n  Food: Bratwurst and Sauerkraut\n  Currency: Euro", additional_kwargs={}, response_metadata={'finish_reason': 'stop', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_7c2c459cd0', 'service_tier': 'default', 'model_provider': 'openai'}, id='lc_run--01a18775-7841-7570-9354-d0a43d122567', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 153, 'output_tokens': 35, 'total_tokens': 188, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [22]:
chain.invoke({"country": "Turkey"})

AI: 
  Here's what I know:
  Capital: Ankara
  Language: Turkish
  Food: Kebab and Baklava
  Currency: Turkish Lira
  

AIMessage(content="AI: \n  Here's what I know:\n  Capital: Ankara\n  Language: Turkish\n  Food: Kebab and Baklava\n  Currency: Turkish Lira\n  ", additional_kwargs={}, response_metadata={'finish_reason': 'stop', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_7c2c459cd0', 'service_tier': 'default', 'model_provider': 'openai'}, id='lc_run--01a18776-8dcc-72b0-852e-bc67b1b90ed7', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 153, 'output_tokens': 36, 'total_tokens': 189, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

# FewShotChatMessagePromptTemplate

In [23]:
from langchain_core.prompts.few_shot import FewShotChatMessagePromptTemplate

In [24]:
examples = [
  {
    "country": "France",

    "answer": """
      Here is what I know:
      Capital: Paris
      Language: French
      Food: Wine and Cheese
      Currency: Euro
      """,
  },
  {
    "country": "Italy",
    "answer": """
      I know this:
      Capital: Rome
      Language: Italian
      Food: Pizza and Pasta
      Currency: Euro
      """,
  },
  {
    "country": "Greece",
    "answer": """
      I know this:
      Capital: Athens
      Language: Greek
      Food: Souvlaki and Feta Cheese
      Currency: Euro
      """,
  },    
]

In [ ]:
example_prompt = ChatPromptTemplate.from_messages([
    ("human", "What do you know abount {country}?"),  # <- 위 examples 와 동일한 key 로!
    ("ai", "{answer}"),
])

example_prompt = FewShotChatMessagePromptTemplate(
    example_prompt = example_prompt,
    examples=examples,
    # suffix= 등은 필요없다.
)

